# Artemus GPU Training — Foundation Pretraining, Code Adaptation, and Human Teaching

This notebook is the reproducible cloud-GPU entry point for the Artemus research project. It starts a **new from-scratch foundation-pretraining experiment** on a bounded FineWeb-Edu sample. It does not resume the earlier 16k-step task-SFT checkpoint, because that checkpoint was trained on a narrow instruction/task mixture rather than broad next-token pretraining.

**Default first run:** pinned FineWeb-Edu `sample-10BT` stream, up to 20,000 accepted documents, deterministic 1% holdout, 4k SentencePiece vocabulary, compact GQA/RoPE model, 2,000 optimizer steps, mixed precision when CUDA supports it, and periodic checkpoints.

FineWeb-Edu declares ODC-By 1.0. Its Common Crawl origins and dataset-card privacy caveats still require review; regex filters here are not a complete privacy audit. Do not redistribute the resulting checkpoint until the project has reviewed the applicable terms and attribution requirements.

See `docs/training/continuous-training-programme.md` before expanding the corpus or promoting a checkpoint.

## 1. Setup — turn on a GPU

Run this notebook in Kaggle Notebooks or Google Colab. Select a GPU accelerator, enable Internet access for dataset downloads, and check that a CUDA GPU was actually allocated. Free GPU access is best-effort and can be interrupted; checkpoints are written at evaluation intervals.


In [ ]:
import torch
from pathlib import Path
import os, subprocess, json, shutil

if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU was allocated. Enable a GPU accelerator in Notebook settings, or use a CUDA VM.')

print('GPU:', torch.cuda.get_device_name(0))
print('CUDA:', torch.version.cuda)
print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
print('BF16 supported:', torch.cuda.is_bf16_supported())


## 2. Fetch the project and install the data tools

The notebook uses the public `main` branch as its code source. For a strict reproduction, copy the printed commit SHA into your experiment record and rerun with that exact repository commit.


In [ ]:
REPO_URL = 'https://github.com/FAHTHIZ/Artemus.git'
WORK_ROOT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/content')
WORKDIR = WORK_ROOT / 'Artemus-training'

# Never delete an existing workspace. Reuse only a clean clone of this repository.
if WORKDIR.exists():
    if not (WORKDIR / '.git').is_dir():
        raise RuntimeError(f'Workspace already exists and is not a Git clone: {WORKDIR}')
    origin = subprocess.check_output(
        ['git', 'remote', 'get-url', 'origin'], cwd=WORKDIR, text=True
    ).strip().removesuffix('.git').rstrip('/')
    expected_origin = REPO_URL.removesuffix('.git').rstrip('/')
    if origin.lower() != expected_origin.lower():
        raise RuntimeError(f'Existing workspace has a different Git origin: {origin}')
    dirty = subprocess.check_output(
        ['git', 'status', '--porcelain'], cwd=WORKDIR, text=True
    ).strip()
    if dirty:
        raise RuntimeError('Existing Artemus-training clone has local changes; preserve them and use a clean workspace.')
    subprocess.run(['git', 'pull', '--ff-only'], cwd=WORKDIR, check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(WORKDIR)], check=True)
os.chdir(WORKDIR)
REPO_COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('Artemus commit:', REPO_COMMIT)
subprocess.run(['python', '-m', 'pip', 'install', '--upgrade', 'pip'], check=True)
subprocess.run(['python', '-m', 'pip', 'install', '-e', '.[dev,data]'], check=True)


## 3. Reproducible experiment settings

Keep the first run intentionally bounded. Increase document count and training steps only after the data pipeline, memory use, validation behavior, and checkpoint recovery are verified. `sample-10BT` names the upstream subset; this notebook does **not** download 10 billion tokens.


In [ ]:
DATASET_ID = 'HuggingFaceFW/fineweb-edu'
DATASET_CONFIG = 'sample-10BT'
DATASET_SPLIT = 'train'
DATASET_REVISION = '87f09149ef4734204d70ed1d046ddc9ca3f2b8f9'
DATASET_LICENSE = 'ODC-By-1.0'
LICENSE_REVIEW_NOTE = ('Research pilot only: upstream card declares ODC-By 1.0; '
    'Common Crawl terms and residual PII caveats require separate review before model redistribution.')

MAX_DOCUMENTS = 20_000
VALIDATION_PERCENT = 1.0
MIN_CHARS = 300
MAX_CHARS = 40_000
VOCAB_SIZE = 4096
BLOCK_SIZE = 256
TRAIN_STEPS = 2_000
BATCH_SIZE = 8
GRAD_ACCUM = 2
BASE_LR = 3e-4
PRECISION = 'bf16' if torch.cuda.is_bf16_supported() else 'fp16'

ARTIFACT_ROOT = WORK_ROOT / 'artifacts'
CHECKPOINT_ROOT = WORK_ROOT / 'checkpoints'
DATA_DIR = ARTIFACT_ROOT / 'fineweb_edu'
FINEWEB2_DIR = ARTIFACT_ROOT / 'fineweb2_bangla'
MIX_DIR = ARTIFACT_ROOT / 'foundation_mixture'
TOKENIZER_PREFIX = ARTIFACT_ROOT / 'artemus-foundation-tokenizer'
CHECKPOINT = CHECKPOINT_ROOT / 'artemus-foundation-fineweb-edu.pt'
DATA_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
FINEWEB2_MAX_DOCUMENTS = 5_000
MIX_TRAIN_DOCUMENTS = 10_000
MIX_VALIDATION_DOCUMENTS = 500
FINEWEB_ENGLISH_WEIGHT = 0.85
FINEWEB2_BANGLA_WEIGHT = 0.15
print('Precision:', PRECISION)


### Data-use gate

The notebook records the source licence and review note in its manifest. The note is an audit entry, not legal advice or a substitute for reviewing the dataset card and upstream Common Crawl terms. If you have not accepted that research-use basis, stop before the download cell and choose another reviewed source.


In [ ]:
cmd = [
    'artemus-prepare-hf', '--dataset', DATASET_ID,
    '--config', DATASET_CONFIG, '--split', DATASET_SPLIT,
    '--revision', DATASET_REVISION, '--license', DATASET_LICENSE,
    '--license-review-note', LICENSE_REVIEW_NOTE,
    '--output-dir', str(DATA_DIR),
    '--max-documents', str(MAX_DOCUMENTS),
    '--validation-percent', str(VALIDATION_PERCENT),
    '--min-chars', str(MIN_CHARS), '--max-chars', str(MAX_CHARS),
]
subprocess.run(cmd, check=True)
manifest = json.loads((DATA_DIR / 'manifest.json').read_text())
print(json.dumps(manifest['counts'], indent=2))
print('Accepted docs:', manifest['accepted_documents'])


## 4. Add a bounded Bangla stream and build a mixed foundation corpus\n\nThe Bengali FineWeb2 subset is streamed from its filtered `train` split only. The notebook resolves the current dataset commit once, writes that exact SHA into the source manifest, rejects obvious contact/secret patterns plus conservative phone-like sequences, and never downloads the full Bengali subset. The first model and tokenizer are trained on a deterministic 85/15 document mixture of FineWeb-Edu English and FineWeb2 Bangla. After tokenizer training, the notebook measures actual per-source token shares; they may differ from document weights because Bangla and English tokenize differently.\n\nThe dataset is ODC-By 1.0 and inherits Common Crawl terms. Its upstream maintainers warn that residual PII may remain and phone numbers are not comprehensively removed; our extra pattern filter is not a complete privacy audit. Review sampled documents and the resulting manifest before using or distributing any checkpoint.\n

In [ ]:
from huggingface_hub import HfApi

# Resolve the live dataset revision once; the actual training download is pinned to this SHA.
FINEWEB2_DATASET = 'HuggingFaceFW/fineweb-2'
fineweb2_revision = HfApi().dataset_info(FINEWEB2_DATASET).sha
if not fineweb2_revision or len(fineweb2_revision) < 7:
    raise RuntimeError('Could not resolve an immutable FineWeb2 dataset revision.')
FINEWEB2_LICENSE_NOTE = (
    'Pilot only: FineWeb2 card declares ODC-By 1.0; Common Crawl terms also apply. '
    'Upstream warns residual PII may remain and phone numbers are not comprehensively filtered. '
    'Artemus runs additional regex filtering, but this is not a complete privacy audit.'
)
fineweb2_cmd = [
    'artemus-prepare-hf', '--dataset', FINEWEB2_DATASET,
    '--config', 'ben_Beng', '--split', 'train', '--revision', fineweb2_revision,
    '--license', 'ODC-By-1.0', '--license-review-note', FINEWEB2_LICENSE_NOTE,
    '--output-dir', str(FINEWEB2_DIR), '--max-documents', str(FINEWEB2_MAX_DOCUMENTS),
    '--validation-percent', '1.0', '--min-chars', '300', '--max-chars', '40000',
]
subprocess.run(fineweb2_cmd, check=True)
fineweb2_manifest = json.loads((FINEWEB2_DIR / 'manifest.json').read_text(encoding='utf-8'))
fineweb2_manifest['resolved_revision'] = fineweb2_revision
(FINEWEB2_DIR / 'manifest.json').write_text(
    json.dumps(fineweb2_manifest, indent=2, ensure_ascii=False) + '\n', encoding='utf-8'
)
print('FineWeb2 revision:', fineweb2_revision)
print('Accepted Bangla documents:', fineweb2_manifest['accepted_documents'])


### Build the foundation mixture\n\nWe keep the split boundaries created by each source, choose deterministic document quotas, deduplicate across sources, and retain each record's original provenance. The 85/15 settings are a starting hypothesis—not a sacred recipe. Inspect the post-tokenizer shares before training and tune them from measured validation results.\n

In [ ]:
mixture_cmd = [
    'artemus-prepare-mixture', '--output-dir', str(MIX_DIR),
    '--train-documents', str(MIX_TRAIN_DOCUMENTS),
    '--validation-documents', str(MIX_VALIDATION_DOCUMENTS),
    '--seed', '1337',
    '--source', 'fineweb-edu', str(DATA_DIR / 'train.jsonl'), str(DATA_DIR / 'validation.jsonl'), str(FINEWEB_ENGLISH_WEIGHT),
    '--source', 'fineweb2-ben_Beng', str(FINEWEB2_DIR / 'train.jsonl'), str(FINEWEB2_DIR / 'validation.jsonl'), str(FINEWEB2_BANGLA_WEIGHT),
]
subprocess.run(mixture_cmd, check=True)
mixture_manifest = json.loads((MIX_DIR / 'manifest.json').read_text(encoding='utf-8'))
print('Train mixture:', json.dumps(mixture_manifest['train'], indent=2, ensure_ascii=False))
print('Validation mixture:', json.dumps(mixture_manifest['validation'], indent=2, ensure_ascii=False))


## 4. Train a tokenizer on training text only

SentencePiece sees only the curated training documents, not validation. The model uses byte fallback and retains the dedicated chat control tokens for later supervised fine-tuning.


In [ ]:
from artemis.tokenizer import SentencePieceTokenizer
tokenizer = SentencePieceTokenizer.train(
    MIX_DIR / 'train.txt', TOKENIZER_PREFIX, vocab_size=VOCAB_SIZE
)
print('Tokenizer vocab:', tokenizer.vocab_size)
print('Tokenizer model:', TOKENIZER_PREFIX.with_suffix('.model'))


# Measure the real token mixture with the final tokenizer; document ratios are only a proxy.
from collections import Counter
mixture_token_counts = Counter()
for line in (MIX_DIR / 'train.jsonl').open('r', encoding='utf-8'):
    if line.strip():
        row = json.loads(line)
        mixture_token_counts[row['mixture']['source_id']] += len(tokenizer.encode(row['text']))
mixture_token_total = sum(mixture_token_counts.values())
mixture_token_shares = {
    key: value / mixture_token_total
    for key, value in mixture_token_counts.items()
}
mixture_manifest_path = MIX_DIR / 'manifest.json'
mixture_manifest = json.loads(mixture_manifest_path.read_text(encoding='utf-8'))
mixture_manifest['tokenizer_measurement'] = {
    'tokenizer': tokenizer.name,
    'vocab_size': tokenizer.vocab_size,
    'train_tokens_by_source': dict(mixture_token_counts),
    'train_token_share_by_source': mixture_token_shares,
}
mixture_manifest_path.write_text(json.dumps(mixture_manifest, indent=2, ensure_ascii=False) + '\n', encoding='utf-8')
print('Actual token shares:', mixture_token_shares)


## 5. Train the first foundation-pretraining candidate

This is a fresh random initialization. It intentionally does not load the previous task-SFT checkpoint. JSONL input appends an EOS marker to every accepted document. Gradient accumulation increases the effective batch size without requiring all microbatches to fit simultaneously.


In [ ]:
train_cmd = [
    'artemus-train', '--data', str(MIX_DIR / 'train.jsonl'),
    '--validation-data', str(MIX_DIR / 'validation.jsonl'),
    '--data-manifest', str(MIX_DIR / 'manifest.json'),
    '--document-format', 'jsonl', '--text-field', 'text',
    '--token-cache-dir', str(ARTIFACT_ROOT / 'token-cache'),
    '--tokenizer', 'sentencepiece', '--tokenizer-model', str(TOKENIZER_PREFIX.with_suffix('.model')),
    '--phase', 'foundation_pretraining', '--steps', str(TRAIN_STEPS),
    '--batch-size', str(BATCH_SIZE), '--gradient-accumulation-steps', str(GRAD_ACCUM),
    '--precision', PRECISION, '--block-size', str(BLOCK_SIZE),
    '--layers', '4', '--heads', '4', '--embedding-size', '128', '--embedding-dim', '64',
    '--kv-heads', '2', '--use-rope', '--use-qk-norm', '--use-swiglu',
    '--learning-rate', str(BASE_LR), '--lr-schedule', 'cosine',
    '--min-learning-rate', '0.00003', '--warmup-steps', '100',
    '--eval-interval', '250', '--eval-iters', '8', '--seed', '1337',
    '--device', 'auto', '--output', str(CHECKPOINT),
]
subprocess.run(train_cmd, check=True)
assert CHECKPOINT.is_file() and CHECKPOINT.stat().st_size > 0
print('Checkpoint bytes:', CHECKPOINT.stat().st_size)


## 6. Basic generation and dialogue health check

These runs check that the model loads, generates non-empty outputs, and can handle a short multi-turn script. They are not correctness or capability benchmarks; retain the transcript and use protected tests for actual scoring.


In [ ]:
subprocess.run([
    'artemus-dialogue-eval', '--checkpoint', str(CHECKPOINT),
    '--scenarios', 'data/evaluation/dialogues.jsonl',
    '--output-dir', str(ARTIFACT_ROOT / 'dialogue-eval'),
    '--max-new-tokens', '64', '--top-k', '1', '--temperature', '1.0',
    '--seed', '1337', '--device', 'auto',
], check=True)
dialogue_summary = json.loads((ARTIFACT_ROOT / 'dialogue-eval' / 'summary.json').read_text())
print(json.dumps(dialogue_summary, indent=2))


## 7. Optional code-domain continuation with OSCR

OSCR scripts are **not** under one universal licence. Each repository manifest preserves the source licence and the evidence used to confirm it. Only select repository manifest paths whose exact SPDX licence appears in the tool's allowlist (MIT, Apache-2.0, BSD, ISC, CC0, etc.). Exclude NC/SA/unknown/compound licences unless a separate policy has approved them. OSCR recommends downloading only selected manifests and blocks, not crawling its site.

Add exact paths from the OSCR dataset's `manifests/.../*.json` tree below. Leave the list empty to skip this stage. The dataset revision is pinned in the script.


In [ ]:
OSCR_MANIFEST_PATHS = []  # e.g. ['manifests/ab/<repository-manifest>.json']
OSCR_DIR = ARTIFACT_ROOT / 'oscr'
OSCR_MIX_DIR = ARTIFACT_ROOT / 'code_replay_mixture'
if OSCR_MANIFEST_PATHS:
    oscr_cmd = ['artemus-prepare-oscr', '--output-dir', str(OSCR_DIR),
                '--dataset-revision', 'b0e3fadb276678880f7f3d764308fbebd3bcd044',
                '--max-files', '10000']
    for manifest_path in OSCR_MANIFEST_PATHS:
        oscr_cmd.extend(['--hub-manifest-path', manifest_path])
    subprocess.run(oscr_cmd, check=True)

    # Retain the general multilingual foundation mixture during code adaptation.
    code_mix_cmd = [
        'artemus-prepare-mixture', '--output-dir', str(OSCR_MIX_DIR),
        '--train-documents', '10000', '--validation-documents', '500',
        '--seed', '1338',
        '--source', 'general-replay', str(MIX_DIR / 'train.jsonl'), str(MIX_DIR / 'validation.jsonl'), '0.85',
        '--source', 'python-code', str(OSCR_DIR / 'train.jsonl'), str(OSCR_DIR / 'validation.jsonl'), '0.15',
    ]
    subprocess.run(code_mix_cmd, check=True)
    parent_state = torch.load(CHECKPOINT, map_location='cpu', weights_only=True)
    code_target_step = int(parent_state['step']) + 3000
    CODE_CHECKPOINT = CHECKPOINT_ROOT / 'artemus-code-adapted.pt'
    subprocess.run([
        'artemus-train', '--data', str(OSCR_MIX_DIR / 'train.jsonl'),
        '--validation-data', str(OSCR_MIX_DIR / 'validation.jsonl'),
        '--data-manifest', str(OSCR_MIX_DIR / 'manifest.json'),
        '--document-format', 'jsonl', '--text-field', 'text',
        '--token-cache-dir', str(ARTIFACT_ROOT / 'oscr-token-cache'),
        '--tokenizer', 'sentencepiece',
        '--resume', str(CHECKPOINT), '--allow-data-change',
        '--phase', 'continued_pretraining', '--steps', str(code_target_step),
        '--batch-size', str(BATCH_SIZE), '--gradient-accumulation-steps', str(GRAD_ACCUM),
        '--precision', PRECISION, '--override-resume-learning-rate',
        '--learning-rate', '0.0001', '--lr-schedule', 'cosine', '--min-learning-rate', '0.00001',
        '--warmup-steps', '100', '--eval-interval', '250', '--eval-iters', '8',
        '--device', 'auto', '--output', str(CODE_CHECKPOINT),
    ], check=True)
    CHECKPOINT = CODE_CHECKPOINT
else:
    print('OSCR stage skipped: no repository manifests selected.')


## 8. Human teaching and supervised fine-tuning

You can talk to the model in a local terminal using `artemus-teach --checkpoint <checkpoint>`. It asks you to correct, approve, or skip each answer. Corrections are saved as human-reviewed candidates; model-generated answers are not included unless you explicitly enable the approval flag. Review the candidate JSONL before using it.

To use a reviewed candidate file as SFT data, upload it as a Kaggle input and run a separate stage with `--response-only --allow-data-change --allow-objective-change`. Do not combine arbitrary unreviewed chat logs with the training corpus.


## 9. Export checkpoint and provenance

Save a Kaggle Notebook version with outputs enabled. In Colab, download the generated ZIP from the file browser. Keep weights and run artifacts out of Git; attach the saved notebook output to the experiment record. The manifest and checkpoint metadata preserve data lineage, but review provenance before public model release.


In [ ]:
EXPORT_DIR = WORK_ROOT / 'artemus-run'
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy2(CHECKPOINT, EXPORT_DIR / CHECKPOINT.name)
shutil.copy2(DATA_DIR / 'manifest.json', EXPORT_DIR / 'fineweb-edu-manifest.json')
shutil.copy2(DATA_DIR / 'provenance.jsonl', EXPORT_DIR / 'fineweb-edu-provenance.jsonl')
shutil.copy2(TOKENIZER_PREFIX.with_suffix('.model'), EXPORT_DIR / 'tokenizer.model')
dialogue_summary_path = ARTIFACT_ROOT / 'dialogue-eval' / 'summary.json'
if dialogue_summary_path.exists():
    shutil.copy2(dialogue_summary_path, EXPORT_DIR / 'dialogue-summary.json')
EXPORT_ZIP = shutil.make_archive(str(WORK_ROOT / 'artemus-run'), 'zip', EXPORT_DIR)
print('Export directory:', EXPORT_DIR)
print('Export archive:', EXPORT_ZIP)
print('Files:', [p.name for p in EXPORT_DIR.iterdir()])


## 10. Next steps

1. Compare the new foundation checkpoint against the old task-SFT checkpoint using the same evaluation set and decoding budget.
2. Add a curated Bangla stream only after reviewing its component/source rights. B-CORE's dataset-level CC-BY-4.0 label does not automatically clear every underlying source.
3. Add verified math/code SFT tasks using official training splits only; keep validation/test splits protected.
4. Add human-preference data and verified outcome optimization only after the basic supervised and tool-verification pipeline is stable.
5. Never train indefinitely on one sample: advance to a new pinned data release or stop when held-out validation plateaus or regresses.
